# Глава 5. Предварительное обучение на неразмеченных данных

In [1]:
pip install matplotlib numpy tiktoken torch tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from importlib.metadata import version

pkgs = ["matplotlib", 
        "numpy", 
        "tiktoken", 
        "torch",
        "tensorflow" # Для предварительно обученных моделей OpenAI
       ]
for p in pkgs:
    print(f"{p} Версия: {version(p)}")

matplotlib Версия: 3.10.9
numpy Версия: 2.4.4
tiktoken Версия: 0.12.0
torch Версия: 2.12.0
tensorflow Версия: 2.21.0


- В этой главе мы реализуем цикл обучения и код для базовой оценки модели, чтобы провести предварительное обучение большой языковой модели
- В конце мы также загружаем в нашу модель общедоступные предварительно обученные веса от OpenAI

<img src="https://camo.githubusercontent.com/137f57f6192fbcb6627e6ced1b5274c71924774dec18a4dea29b1c156619ef24/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30312e77656270" width=800px>

- Ниже перечислены темы, затронутые в этой главе

<img src="https://camo.githubusercontent.com/01ebc99e37dddc617ba6dd20799f945fd6a562bac8b8abe5a4b82eb491ebbfca/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30322e77656270" width=800px>

&nbsp;
## 5.1 Оценка генеративных текстовых моделей

- В начале этого раздела мы кратко расскажем о том, как инициализировать модель GPT с помощью кода из предыдущей главы
- Затем мы обсудим основные метрики оценки больших языковых моделей
- Наконец, в этом разделе мы применим эти метрики оценки к обучающему и проверочному наборам данных

&nbsp;
### 5.1.1 Использование GPT для генерации текста

- Мы инициализируем модель GPT с помощью кода из предыдущей главы

In [3]:
import torch
from previous_chapters import GPTModel
# Если файл `previous_chapters.py` недоступен локально,
# вы можете импортировать его из пакета PyPI `llms-from-scratch`. 
# Подробнее см.: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# Например,
# from llms_from_scratch.ch04 import GPTModel

GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Размер словаря
    "context_length": 256, # Сокращенная длина контекста (исходное значение: 1024)
    "emb_dim": 768,        # Размерность эмбеддинга
    "n_heads": 12,         # Количество ядер внимания
    "n_layers": 12,        # Количество слоев
    "drop_rate": 0.1,      # Коэффициент дропаута
    "qkv_bias": False      # Смещение в сторону значений ключей запроса
}

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval();  # Отключите дропаут во время логического вывода

- Мы используем дропаут 0,1, но в настоящее время довольно часто обучают большие языковые модели без дропаута
- В современных больших языковых моделях также не используются векторы смещения в слоях `nn.Linear` для матриц запросов, ключей и значений (в отличие от более ранних моделей GPT). Это достигается за счет установки параметра `"qkv_bias": False`
- Мы уменьшили длину контекста (`context_length`) всего на 256 токенов, чтобы снизить требования к вычислительным ресурсам для обучения модели, в то время как исходная модель GPT-2 со 124 миллионами параметров использовала 1024 токена
    - Это сделано для того, чтобы мы могли следить за примерами кода и выполнять их на своих портативных компьютерах
    - Позже мы также загрузим модель с `context_length` 1024 из предварительно обученных весов.

- Далее мы используем функцию `generate_text_simple` из предыдущей главы для генерации текста
- Кроме того, мы определяем две вспомогательные функции: `text_to_token_ids` и `token_ids_to_text` — для преобразования токенов в текстовое представление и обратно, которые мы будем использовать на протяжении всей главы

<img src="https://camo.githubusercontent.com/0756c65e7e7878cab7b7673bedd3f441cf42f1f67e89b46d9e7ec931e0fffbc3/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30332e77656270" width=800px>

In [4]:
import tiktoken
from previous_chapters import generate_text_simple

def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0) # добавить размер пакета
    return encoded_tensor

def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0) # удалить размер пакета
    return tokenizer.decode(flat.tolist())

start_context = "Every effort moves you"
tokenizer = tiktoken.get_encoding("gpt2")

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(start_context, tokenizer),
    max_new_tokens=10,
    context_size=GPT_CONFIG_124M["context_length"]
)

print("Выводимый текст:\n", token_ids_to_text(token_ids, tokenizer))

Выводимый текст:
 Every effort moves you rentingetic wasnم refres RexMeCHicular stren


- Как мы видим выше, модель не генерирует качественный текст, потому что она еще не обучена
- Как измерить или зафиксировать в числовом выражении, что такое «качественный текст», чтобы отслеживать этот показатель во время обучения?
- В следующем подразделе мы рассмотрим метрики для расчета показателя потерь для сгенерированных результатов, которые можно использовать для оценки прогресса обучения
- В следующих главах, посвященных тонкой настройке больших языковых моделей, мы также рассмотрим дополнительные способы оценки качества модели

&nbsp;
### 5.1.2 Расчет потерь при генерации текста: кросс-энтропия и перплексия

- Предположим, у нас есть тензор `inputs`, содержащий идентификаторы токенов для двух обучающих примеров (строк)
- Соответствующие `inputs`, `targets` содержат желаемые идентификаторы токенов, которые мы хотим, чтобы модель сгенерировала
- Обратите внимание, что `targets` — это `inputs`, сдвинутые на одну позицию

In [5]:
inputs = torch.tensor([[16833, 3626, 6100],   # ["every effort moves",
                       [40,    1107, 588]])   #  "I really like"]

targets = torch.tensor([[3626, 6100, 345  ],  # [" effort moves you",Ф
                        [1107,  588, 11311]]) #  " really like chocolate"]

- Подавая на вход модели данные, мы получаем вектор логитов для двух входных примеров, каждый из которых состоит из 3 токенов
- Каждый токен представляет собой вектор из 50 257 элементов, соответствующий размеру словаря
- Применяя функцию softmax, мы можем преобразовать тензор логитов в тензор той же размерности, содержащий оценки вероятности

In [6]:
with torch.no_grad():
    logits = model(inputs)

probas = torch.softmax(logits, dim=-1) # Вероятность появления каждого токена в словаре
print(probas.shape) # Форма: (размер пакета, количество токенов, размер словаря)

torch.Size([2, 3, 50257])


- На рисунке ниже с использованием очень небольшого набора слов для наглядности показано, как мы преобразуем оценки вероятности обратно в текст

<img src="https://camo.githubusercontent.com/f98790fc96dfefdd3e61533ca406f241a893976c8cb7668afbcec178763c45a0/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30342e77656270" width=800px>

- Ммы можем использовать функцию `argmax`, чтобы преобразовать значения вероятности в предсказанные идентификаторы токенов
- Функция `softmax`, описанная выше, сгенерировала 50 257-мерный вектор для каждого токена. Функция `argmax` возвращает позицию с наибольшим значением вероятности в этом векторе, которая и является предсказанным идентификатором токена

- Поскольку у нас есть 2 входных пакета по 3 токена в каждом, мы получаем 2 на 3 предсказанных идентификатора токенов:

In [7]:
token_ids = torch.argmax(probas, dim=-1, keepdim=True)
print("Идентификаторы токенов:\n", token_ids)

Идентификаторы токенов:
 tensor([[[16657],
         [  339],
         [42826]],

        [[49906],
         [29669],
         [41751]]])


- Если мы расшифруем эти токены, то увидим, что они сильно отличаются от тех, которые мы хотим, чтобы модель предсказывала

In [8]:
print(f"Целевой batch 1: {token_ids_to_text(targets[0], tokenizer)}")
print(f"Фактический batch 1: {token_ids_to_text(token_ids[0].flatten(), tokenizer)}")

Целевой batch 1:  effort moves you
Фактический batch 1:  Armed heNetflix


- Это потому, что модель еще не обучена
- Чтобы обучить модель, нам нужно знать, насколько она далека от правильных прогнозов (целевых значений)

<img src="https://camo.githubusercontent.com/3dee5bf33ad015c683fa2a9a91ae611b263101027fa2f9415934ae1a1c38778e/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30362e77656270" width=800px>

- Вероятности появления токенов, соответствующие целевым индексам, следующие:

In [9]:
text_idx = 0
target_probas_1 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 1:", target_probas_1)

text_idx = 1
target_probas_2 = probas[text_idx, [0, 1, 2], targets[text_idx]]
print("Текст 2:", target_probas_2)

Текст 1: tensor([7.4541e-05, 3.1061e-05, 1.1563e-05])
Текст 2: tensor([1.0337e-05, 5.6776e-05, 4.7559e-06])


 ---

### Код выше

Эта строка выполняет **индексирование многомерного массива (тензора) `probas`** для извлечения вероятностей, соответствующих правильным целевым классам.

### Пошаговый разбор

### 1. `text_idx`
Скалярный индекс, указывающий на конкретный текст в батче.
- `probas[text_idx, ...]` → выбирает двумерный срез формы `(количество_токенов, количество_классов)` для одного текста.

### 2. `[0, 1, 2]`
Список индексов токенов.
- `probas[text_idx, [0, 1, 2], ...]` → выбирает строки только для токенов с индексами 0, 1, 2 (первые три токена).
- После этого измерения получается форма `(3, количество_классов)`.

### 3. `targets[text_idx]`
Одномерный массив формы `(количество_токенов,)`, содержащий **истинные метки классов** для каждого токена в тексте `text_idx`.
- `targets[text_idx]` → вектор правильных классов для всего текста.
- Но поскольку на предыдущем шаге мы выбрали только токены `[0, 1, 2]`, здесь **неявно тоже берутся первые три элемента** этого вектора (благодаря broadcasting/advanced indexing).

### 4. Полное индексирование
```python
probas[text_idx, [0, 1, 2], targets[text_idx]]
```
NumPy/PyTorch выполняет **advanced indexing**:
- Для каждого из выбранных токенов (0, 1, 2) извлекается вероятность того класса, который указан в `targets` для этого же токена.
- Фактически это эквивалентно:
```python
[
    probas[text_idx, 0, targets[text_idx][0]],  # вер-ть правильного класса для токена 0
    probas[text_idx, 1, targets[text_idx][1]],  # вер-ть правильного класса для токена 1
    probas[text_idx, 2, targets[text_idx][2]]   # вер-ть правильного класса для токена 2
]
```

### Результат

**`target_probas_2`** — одномерный массив из трёх чисел (вероятностей), показывающих, насколько модель была уверена в **правильных** классах для первых трёх токенов конкретного текста.

Это часто используется для:
- анализа уверенности модели в правильных ответах,
- вычисления **confidence** срезов,
- поиска сложных примеров (где верная вероятность мала),
- отладки качества предсказаний на уровне отдельных токенов.

---

- Мы хотим максимизировать все эти значения, приблизив их к вероятности 1.
- В математической оптимизации проще максимизировать логарифм показателя вероятности, чем сам показатель вероятности. Лекция с более подробным описанием: [L8.2 Функция потерь логистической регрессии](https://www.youtube.com/watch?v=GxJe0DZvydM)

In [12]:
# Вычислить логарифм всех вероятностей токенов
log_probas = torch.log(torch.cat((target_probas_1, target_probas_2)))
print(log_probas)

tensor([ -9.5042, -10.3796, -11.3677, -11.4798,  -9.7764, -12.2561])


- Далее мы вычисляем среднюю логарифмическую вероятность:

In [13]:
# Рассчитайте среднюю вероятность для каждого токена
avg_log_probas = torch.mean(log_probas)
print(avg_log_probas)

tensor(-10.7940)


- Цель состоит в том, чтобы сделать эту среднюю логарифмическую вероятность как можно более высокой за счет оптимизации весовых коэффициентов модели
- Из-за логарифмической функции максимально возможное значение равно 0, а мы пока далеки от этого значения

- В глубоком обучении вместо максимизации средней логарифмической вероятности принято минимизировать *отрицательное* значение средней логарифмической вероятности. В нашем случае вместо того, чтобы максимизировать -10.7940, чтобы оно приблизилось к 0, в глубоком обучении мы минимизируем -10.7940, чтобы оно приблизилось к 0
- Отрицательное значение -10.7940, то есть -10.7940, в глубоком обучении также называют кросс-энтропийной потерей

In [14]:
neg_avg_log_probas = avg_log_probas * -1
print(neg_avg_log_probas)

tensor(10.7940)


В PyTorch уже реализована функция `cross_entropy`, которая выполняет описанные выше действия

<img src="https://camo.githubusercontent.com/3d6fb2ee91bebb246351570506a344d7d579fc161084faec6856c0659c815452/68747470733a2f2f73656261737469616e72617363686b612e636f6d2f696d616765732f4c4c4d732d66726f6d2d736372617463682d696d616765732f636830355f636f6d707265737365642f30372e77656270" width=800px>

- Прежде чем применить функцию `cross_entropy`, давайте проверим форму логитов и целевых значений

In [15]:
# Логиты имеют форму (batch_size, num_tokens, vocab_size)
print("Форма логитов:", logits.shape)

# Цель имеет форму (batch_size, num_tokens)
print("Целевая форма:", targets.shape)

Форма логитов: torch.Size([2, 3, 50257])
Целевая форма: torch.Size([2, 3])


- Для функции `cross_entropy` в PyTorch мы хотим сгладить эти тензоры, объединив их по размерности пакета:

In [16]:
logits_flat = logits.flatten(0, 1)
targets_flat = targets.flatten()

print("Сглаженные логиты:", logits_flat.shape)
print("Сглаженные цели:", targets_flat.shape)

Сглаженные логиты: torch.Size([6, 50257])
Сглаженные цели: torch.Size([6])


- Обратите внимание, что целевыми значениями являются идентификаторы токенов, которые также представляют собой позиции индексов в тензорах логитов, которые мы хотим максимизировать
- Функция `cross_entropy` в PyTorch автоматически применяет функцию `softmax` и вычисляет логарифмическую вероятность для тех индексов токенов в логитах, которые нужно максимизировать

In [17]:
loss = torch.nn.functional.cross_entropy(logits_flat, targets_flat)
print(loss)

tensor(10.7940)


- Понятие, связанное с кросс-энтропийной потерей, — это перплексия большой языковой модели
- Перплексия — это просто экспоненциальная функция от кросс-энтропийной потери

In [18]:
perplexity = torch.exp(loss)
print(perplexity)

tensor(48725.8203)


- Показатель перплексии часто считается более интерпретируемым, поскольку его можно рассматривать как эффективный размер словаря, в котором модель не уверена на каждом этапе (в приведенном выше примере это 48 725 слов или токенов)
- Другими словами, перплексия показывает, насколько хорошо распределение вероятностей, предсказанное моделью, соответствует реальному распределению слов в наборе данных
- Как и в случае с логарифмической функцией потерь, чем ниже перплексия, тем ближе предсказания модели к реальному распределению